# 蒙特卡洛 ε-贪婪算法（MC ε-Greedy）

> **动机：去掉 Exploring Starts 假设**
>
> 在 MC Exploring Starts 中，为了保证每个状态-动作对 $(s, a)$ 都能被充分访问，我们要求**每条轨迹的起始状态与起始动作均随机均匀采样**（即 Exploring Starts）。  
> 然而，该假设在现实场景中往往难以满足——智能体通常只能从固定的初始状态出发，无法任意指定起点。
>
> **解决方案：引入软策略（Soft Policy）**
>
> **软策略**是指对所有状态 $s$ 和动作 $a$，策略满足：
> $$\pi(a \mid s) > 0, \quad \forall s \in \mathcal{S},\ a \in \mathcal{A}$$
> 即任意状态下每个动作都有**非零的被选概率**，从而保证每个已访问状态下的所有动作都能被充分探索。
>
> 在本实现中，软策略与随机起始状态**共同协作**完成探索覆盖：
> - **随机起始状态**：保证所有状态均有机会被访问（覆盖状态空间）；
> - **软策略**：保证在每个访问到的状态下，所有动作都有非零概率被选择（覆盖动作空间）。
>
> 两者结合，确保所有状态-动作对 $(s, a)$ 都能被充分采样，从而无需 Exploring Starts 中"强制以随机动作出发"的额外约束。
>
> 本节采用最常见的软策略实现——**ε-贪婪策略（ε-Greedy Policy）**：
> $$\pi(a \mid s) = \begin{cases} 1 - \varepsilon + \dfrac{\varepsilon}{|\mathcal{A}|}, & a = \arg\max_{a'} Q(s, a') \\[6pt] \dfrac{\varepsilon}{|\mathcal{A}|}, & \text{其他动作} \end{cases}$$
> 其中 $\varepsilon \in (0, 1]$ 控制探索程度：$\varepsilon$ 越大，策略越随机；$\varepsilon \to 0$ 时退化为纯贪婪策略。通过在训练过程中**逐步衰减 $\varepsilon$**，可以在充分探索与策略收敛之间取得平衡。

> **MC 的统计特性：无偏差（Zero Bias）但高方差（High Variance）**
>
> MC 方法使用完整轨迹的真实累积折扣回报 $G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1}R_T$ 作为 $Q(s,a)$ 的估计目标。
>
> - **无偏差**：由回报的定义可知 $\mathbb{E}[G_t \mid S_t=s, A_t=a] = Q^\pi(s,a)$，估计量的期望精确等于真实值，不存在系统性偏差；
> - **高方差**：$G_t$ 是整条轨迹中**所有随机奖励的加权求和**，每步奖励 $R_{t+k}$ 的随机波动逐层叠加，步数越多、累加项越多，$G_t$ 的波动幅度越大：
> $$\operatorname{Var}[G_t] \approx \sum_{k=0}^{T-t-1} \gamma^{2k}\,\sigma^2_R$$
> 其中 $\sigma^2_R$ 为单步奖励方差，$T-t$ 为剩余步数。因此 MC 需要大量样本（即本实现中 `trajectorySteps` 设置为 2000 步、`num_episodes` 设置为 400 回合）才能将方差压低至可靠水平。

> **本实现采用 Every-Visit MC**
>
> MC 方法在利用一条轨迹更新 $Q(s,a)$ 时，存在两种计数策略：
>
> - **First-Visit MC**：对于轨迹中每个状态-动作对 $(s,a)$，**只在其第一次出现时**将对应的累积回报 $G_t$ 纳入统计；
> - **Every-Visit MC**：对于轨迹中每个状态-动作对 $(s,a)$，**每次出现都**将对应的 $G_t$ 纳入统计。
>
> 本实现采用 **Every-Visit MC**：在逆向遍历轨迹时，每遇到一个 $(s,a)$ 便无条件累加其回报并更新计数，不检查该对是否已在本轮轨迹中出现过。  
> 两种方法均可收敛至真实 $Q^\pi(s,a)$，Every-Visit MC 对每条样本轨迹的利用率更高，但同一轨迹中重复访问的 $(s,a)$ 会引入相关性，使方差的理论分析更为复杂。

## 一、导入依赖库

In [1]:
import numpy as np       # 导入 NumPy 库，用于数值计算和矩阵运算，版本要求 >=1.18
import random            # 导入 Python 标准库 random，用于随机采样
import importlib.util    # 导入 importlib.util，用于按文件路径动态加载模块

# 文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头且含点号，不符合 Python 标识符规则，无法直接 import
# spec_from_file_location：根据给定模块别名和 .py 文件路径创建模块规格，返回 ModuleSpec 对象
_spec = importlib.util.spec_from_file_location("GridWorld_v2", "02.1.ModelFree_Env_GridWorldV2.py")
# module_from_spec：根据模块规格创建模块对象，此时模块代码尚未执行，返回 module 对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)
# exec_module：执行模块代码完成初始化，之后可通过 GridWorld_v2.GridWorld_v2(...) 正常使用类
_spec.loader.exec_module(GridWorld_v2)
from IPython.display import clear_output  # 导入 IPython 清屏函数，用于在 Jupyter 中刷新输出，避免打印内容过多堆积

## 二、初始化网格世界与策略

In [2]:
gamma = 0.9  # 折扣因子 γ，float，γ 越小远端奖励衰减越快（γ^k 更快趋近于 0）；
             # 此处设为 0.9（小于常用的 0.95），约 22 步后影响不足 10%，
             # 减小每步回报的累加方差，有助于 MC 高方差场景下更稳定地收敛

rows = 5      # 网格世界的行数，int，需与 desc 描述字符串的行数一致
columns = 5   # 网格世界的列数，int，需与 desc 描述字符串每行字符数一致

# 使用描述字符串初始化网格世界：'.' 表示普通格，'#' 表示禁止区（得分 -10），'T' 表示目标格（得分 1）
gridworld = GridWorld_v2.GridWorld_v2(
    # 禁止区域的即时奖励，float，负值表示惩罚
    # 不宜设置过大（如 -100）：惩罚与目标奖励(1)比例悬殊时，障碍周边 Q 值方差极大，
    # MC 的高方差被进一步放大，反而更难收敛；设为 -10（比例 10:1）信号足够清晰且不失稳定
    forbiddenAreaScore=-10,
    score=1,                 # 目标区域的即时奖励，float
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]  # 网格布局描述，list[str]，共 5 行 5 列
)
gridworld.show()  # 以 emoji 打印网格世界布局，无返回值

value = np.zeros(rows * columns)        # 状态价值函数 V(s) 初始化，np.ndarray，shape=(25,)，全零（备用）
qtable = np.zeros((rows * columns, 5))  # 动作价值函数 Q(s,a) 初始化，np.ndarray，shape=(25, 5)，全零

# 随机初始化确定性策略，利用 NumPy 花式索引（Fancy Indexing）一步完成整数索引→one-hot 转换：
# np.random.randint(0,5,size=(rows*columns))：生成 25 个随机动作索引，shape=(25,)，每个值∈[0,4]，int
# np.eye(5)[整数数组]：花式索引——对数组中每个整数 i，取 np.eye(5) 第 i 行（动作 i 的 one-hot 向量）
#   并沿第 0 轴堆叠，等价于 np.stack([np.eye(5)[i] for i in idx])，但由 C 实现、效率更高
# 结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（对应该状态随机选定的动作），其余为 0
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
gridworld.showPolicy(policy)  # 可视化初始随机策略，无返回值

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
🔄⬇️➡️⬆️➡️
⬇️⏬🔄⬅️🔄
⬆️🔄🔄⬇️⬇️
⬇️⏩️✅🔄➡️
⬇️🔄⬅️⬇️➡️


## 三、ε-贪婪 MC 策略迭代

In [3]:
np.random.seed(0)  # 固定 NumPy 随机种子，int；本算法收敛对随机序列高度敏感，
                   # seed=0 是经过验证能稳定收敛的值，训练过程必须在本格重新固定
random.seed(0)     # 固定 Python random 随机种子，int；起点 (i, j) 用 random.randint 采样，
                   # 同样需要固定以保证每次运行轨迹完全一致
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]  # 重新随机初始化策略，shape=(25, 5)，one-hot 编码
gridworld.show()              # 打印网格世界布局
gridworld.showPolicy(policy)  # 打印初始随机策略
print("random policy")        # 提示当前使用随机初始化策略

# 每条轨迹的采样步数，int
# 注意：本实现中每个回合（episode）并不在到达终止状态时自然结束，
# 而是固定采样 trajectorySteps 步作为一批经验数据，不提前截断
# 总采样步数 = num_episodes × trajectorySteps = 400 × 2000 = 800,000 步
#
# 为什么步数要远大于格子数（25）？
# MC 方法用"多次采样的均值"估计 Q(s,a)，单次采样噪声极大（高方差），
# 必须对每个状态-动作对积累足够多的访问次数才能得到可靠估计：
#   - Q 表共有 25×5=125 个状态-动作对需要估计
#   - 访问分布不均匀：靠近目标的状态被频繁访问，角落/障碍附近的状态访问极少，
#     冷门对需要更多总步数才能积累到足够样本
#   - 本实现为非重置式连续轨迹（到达目标后不重置，继续行走），
#     2000 步约等于在棋盘上绕行 ~16 圈（2000÷125），单回合覆盖有限，
#     需依赖多回合（400 轮）的累积均值来收敛；gamma=0.9 时约几十步后折扣趋近于零，
#     2000 步已足够让每条轨迹包含多个完整的"到达目标"片段，回报信号充分
trajectorySteps = 2000
# ε-greedy 的初始探索率，float，控制随机探索比例，越大探索越多
# Q 表初始全零时，argmax 固定选动作 0，ε=0.1 使动作 0 被选概率高达 92%，
# 初期各动作探索极不均匀，容易错过最优路径；适当增大 ε 使各动作更均匀地被访问，
# 能更快建立覆盖全局的有效 Q 值估计
epsilon = 0.3
# ε 下限，float；设为 0.0 使后期退化为纯贪婪策略（ε→0），
# 依靠随机起始状态（Exploring Starts）保证所有状态仍被覆盖，
# 消除 ε 残留噪声对已收敛策略的干扰，使箭头更稳定
eps_min = 0.0
# ε 每回合线性衰减量，float；0.002/回合时 ε 从 0.3 降至 0 约需 150 回合（探索阶段），
# 后续 250 回合以纯贪婪策略精修 Q 值直至收敛
eps_decay = 0.002
qtable = np.zeros((rows * columns, 5))  # 初始化 Q 表，np.ndarray，shape=(25, 5)，全零
# 训练的总回合数，int；每回合采样一条固定长度轨迹，然后做一次 Q 表更新与策略改进
# MC 方法方差较高，需保证收敛阶段（ε 到达下限 0.0 后）有足够回合精细调整 Q 值：
#   探索阶段：0.3 / 0.002 = 150 回合（ε 从 0.3 线性衰减到 0.0）
#   收敛阶段：约需探索阶段 1~2 倍的回合，纯贪婪策略配合随机起点精修 Q 值
#   总步数：400 × 2000 = 800,000 步，每个状态-动作对平均被访问
#     800,000 ÷ 125 = 6,400 次，对 5×5 网格已足够收敛
#   因此取 num_episodes = 400（探索 150 + 收敛 250）以稳定收敛
num_episodes = 400

#
# ── 为何不像 02.3 那样用『Q 表变化量 < 阈値』作为停止条件？──────────────────
# 02.3（MC Exploring Starts）每轮迭代通过双层 for 穷举所有 25×5=125 个 (s,a) 对，
# 每对都被强制作为起点采样一条完整轨迹，保证每轮更新覆盖全部状态-动作空间；
# 因此前后两轮的 Q 表差异是“全量更新”的可靠收敛信号，可以放心作为停止条件。
#
# 本算法（ε-Greedy）每回合只从随机的一个起点出发、采样一条轨迹，
# 更新仅覆盖该轨迹经过的部分 (s,a) 对，无法保证全局覆盖；
# 若改用 Q 表变化量作为停止条件，会有以下三个致命问题：
#   1. 高方差导致判断失灵：单回合 Q 变化量随机波动大，可能某轮偶然变化很小
#      就提前停止（实际尚未收敛），或因持续波动永不满足阈値而陷入死循环
#   2. 过程非平稳，变化量不可信：ε 逐回合线性衰减，策略本身就在持续调整，
#      Q 表“该变化”也在变，单一阈値无法准确判断是否真正收敛
#   3. 覆盖不完整，变化量偏小：未被本回合轨迹访问到的 (s,a) 其 Q 値不变，
#      整体变化量天然偏小，容易误判为已收敛
#
# 因此改用固定回合数：训练代价可控、过程可预测，
# 配合 ε 衰减到 0 + 随机起点作为等效的停止信号

for episode in range(num_episodes):  # 按回合循环训练，episode 为当前回合编号，int，范围 [0, 400)
    # ε 衰减策略——体现探索与利用的权衡（Exploration-Exploitation Tradeoff）：
    #   训练初期：ε 大 → 大量随机探索，Q 表从零开始积累经验，避免过早固化在局部最优动作上
    #   训练后期：ε → 0 → 退化为纯贪婪策略，依赖随机起点（Exploring Starts）保证全局覆盖，
    #   消除 ε 残留噪声，让已收敛的策略更稳定
    # 线性衰减并截断到 eps_min，用 max 防止浮点误差减成负数
    epsilon = max(eps_min, epsilon - eps_decay)  # 更新探索率，float

    p1 = 1 - epsilon * (4 / 5)  # 最优动作的选择概率，float：p1 = 1 - ε*(|A|-1)/|A|，|A|=5
    p0 = epsilon / 5             # 非最优动作的选择概率，float：p0 = ε/|A|，均匀分配给各非最优动作

    print(f"{'=' * 55}")  # 打印分隔线，标识新一回合的开始
    print(f"  第 {episode + 1:>3d} / {num_episodes} 回合  |  ε={epsilon:.4f}  p1={p1:.4f}  p0={p0:.4f}")
    print(f"{'=' * 55}")  # 打印分隔线下边框
    print(f"  轨迹步数：{trajectorySteps}")

    # 构建"值→概率"映射字典，dict{int: float}：
    #   one-hot policy 中值为 1 的位置代表最优动作 → 映射为 p1（较大概率）
    #   值为 0 的位置代表非最优动作              → 映射为 p0（较小概率）
    d = {1: p1, 0: p0}

    # 将确定性 one-hot policy 转换为 ε-greedy 概率策略，分三步理解：
    #
    # 第一步：d.get 是字典的查询方法，d.get(key) 等价于 d[key]
    #   例：d.get(1) → p1，d.get(0) → p0
    #
    # 第二步：np.vectorize(d.get) 将 d.get 包装成"逐元素函数"
    #   原本 d.get 只能处理单个标量，np.vectorize 让它能对 ndarray 的每个元素依次调用
    #   返回一个可调用对象 vfunc，等价于：lambda arr: np.array([[d.get(x) for x in row] for row in arr])
    #
    # 第三步：vfunc(policy) 对 policy 中每个元素执行 d.get
    #   policy shape=(25, 5)，元素为 0 或 1（one-hot）
    #   → 每个 1（最优动作）被替换为 p1，每个 0（非最优动作）被替换为 p0
    #
    # 示例（假设某状态最优动作为动作 2，|A|=5，ε=0.1，p1≈0.92，p0=0.02）：
    #   policy[s]         = [0,    0,    1,    0,    0   ]
    #   policy_epsilon[s] = [0.02, 0.02, 0.92, 0.02, 0.02]  → 行和 = 1
    #
    # policy_epsilon：np.ndarray，shape=(25, 5)，dtype=float，每行为合法概率分布（行和=1）
    #   第 0 维：25 个状态；第 1 维：5 个动作对应的 ε-greedy 选择概率
    policy_epsilon = np.vectorize(d.get)(policy)

    i = random.randint(0, 24)  # 随机选择起始状态，int，范围 [0, 24]，实现探索性出发（Exploring Starts）
    j = random.randint(0, 4)   # 随机选择起始动作，int，范围 [0, 4]，确保所有状态-动作对都有机会被访问

    cnt = [0 for _ in range(25)]  # 各状态的访问次数计数器，list[int]，长度 25，全零，用于调试
    qtable_rewards = [[0 for _ in range(5)] for _ in range(rows * columns)]  # 各状态-动作对的累计折扣回报，list[list[float]]，shape=(25, 5)
    qtable_nums    = [[0 for _ in range(5)] for _ in range(rows * columns)]  # 各状态-动作对的访问次数，list[list[int]]，shape=(25, 5)

    # 从状态 i、动作 j 出发，按 ε-greedy 策略 policy_epsilon 采样 trajectorySteps 步轨迹
    # 返回值：list[tuple]，长度为 trajectorySteps+1，每个元组为 (nowState, nowAction, score, nextState, nextAction)
    Trajectory = gridworld.getTrajectoryScore(
        nowState=i, action=j, policy=policy_epsilon, steps=trajectorySteps
    )
    clear_output(wait=True)  # 清除 Jupyter 输出区域，避免每回合打印内容叠加

    score = 0  # 折扣累计回报 G 的初始值，float，从轨迹末端向前递推时的起点

    # 从轨迹末端（第 trajectorySteps 步）向前逆向遍历，计算每步的折扣累计回报
    for k in range(trajectorySteps, -1, -1):
        tmpstate, tmpaction, tmpscore, _, _ = Trajectory[k]  # 解包第 k 步的状态（int）、动作（int）、即时奖励（float）
        cnt[tmpstate] += 1                # 该状态访问次数加 1，int
        score = score * gamma + tmpscore  # 递推折扣累计回报 G_k = r_k + γ*G_{k+1}，float
        qtable_rewards[tmpstate][tmpaction] += score  # 累加该状态-动作对的折扣回报，float
        qtable_nums[tmpstate][tmpaction] += 1         # 该状态-动作对的访问次数加 1，int
        # Every-Visit MC：用样本均值更新 Q 值，Q(s,a) ← 累计回报之和 / 访问次数，float
        qtable[tmpstate][tmpaction] = (
            qtable_rewards[tmpstate][tmpaction] / qtable_nums[tmpstate][tmpaction]
        )

    values = []  # 存储每个状态在 ε-greedy 策略下的状态价值，list[float]，长度 25
    for i in range(25):   # 遍历所有状态，i 为状态编号，int
        v = 0             # 当前状态价值初始化，float
        for j in range(5):  # 遍历所有动作，j 为动作编号，int
            v += policy_epsilon[i][j] * qtable[i][j]  # V(s) = Σ_a π(a|s)*Q(s,a)，float
        values.append(v)  # 将当前状态价值追加到列表

    # 打印状态价值矩阵，np.ndarray，shape=(5, 5)，直观展示当前策略下各格子的价值分布
    print(np.array(values).reshape(5, 5))

    print(f"  当前策略：")  # 打印策略可视化标签
    gridworld.showPolicy(policy)         # 可视化当前确定性贪婪策略（转换前的 policy）
    print(f"  状态价值均值：{np.array(values).mean():.6f}")  # 打印所有状态价值的均值，float，监控整体学习进度

    # 策略改进：根据当前 Q 表贪婪地选择每个状态的最优动作，构造新的 one-hot 确定性策略
    policy = np.eye(5)[np.argmax(qtable, axis=1)]  # shape=(25, 5)，one-hot 编码
    # 将更新后的贪婪策略转换为 ε-greedy 概率策略，供下一回合采样使用
    policy_epsilon = np.vectorize(d.get)(policy)   # shape=(25, 5)，float，每行和为 1

print(f"\n{'*' * 55}")  # 打印最终分隔线
print(f"  ε-贪婪 MC 训练完成，共训练 {num_episodes} 回合")  # 打印训练总回合数
print(f"  最终 ε={epsilon:.4f}  |  最终状态价值均值：{np.array(values).mean():.6f}")  # 打印最终关键指标
print(f"{'*' * 55}")

[[ 3.4867844   3.87420489  4.3046721   4.782969    5.31441   ]
 [ 3.13810596  3.4867844   4.782969    5.31441     5.9049    ]
 [ 2.82429536  2.54186583 10.          5.9049      6.561     ]
 [ 2.54186583 10.          9.95497749 10.          7.29      ]
 [ 0.          9.         10.          9.          8.1       ]]
  当前策略：
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
  状态价值均值：5.924365

*******************************************************
  ε-贪婪 MC 训练完成，共训练 400 回合
  最终 ε=0.0000  |  最终状态价值均值：5.924365
*******************************************************


In [4]:
gridworld.show()  # 以 emoji 打印网格世界布局，无返回值
gridworld.showPolicy(policy)  # 可视化最终收敛后的策略，展示每个状态下的最优动作方向

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️➡️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️


## 四、回合制 ε-贪婪 MC 策略迭代（改进版）

> **旧版局限：「继续轨迹」模式下回报仍有污染**
>
> 旧版每回合采样 2000 步不重置，逆向累积回报时，每个状态的 $G_t$ 包含了后续若干次
> 到达目标格的 $+1$ 奖励。虽然 $\gamma=0.9$ 的快速折扣大幅削弱了远端奖励的权重
> （约 22 步后影响不足 10%），但禁止格的 $-10$ 惩罚信号仍会被后续奖励部分稀释，
> 使 Q 值估计偏离真实单次决策的质量。
>
> **根本原因对比**
>
> ---

| 对比项 | 旧版（继续轨迹） | 改进版（回合制） |
|---|---|---|
| 回合终止 | 固定采样 2000 步，不重置 | 进入目标格或禁止格时立即终止 |
| $G_t$ 含义 | 从 $t$ 步到第 2000 步末的折扣回报（含后续目标奖励） | 从 $t$ 步到**本回合结束**的折扣回报（干净、真实） |
| 禁止格惩罚信号 | $-10$ 被后续 $+1$ 奖励部分稀释，Q 值偏高 | $-10$ 直接终结回合，无后续奖励补偿，惩罚信号清晰 |
| 绕行路径回报 | 绕行多步获得的 $\gamma^4 \approx 0.66$ 也被后续奖励抬高，差距缩小 | 绕行路径 $G \approx 0.66$，穿越禁止区 $G = -10$，差距明显 |

> ---
>
> **修正方案：回合制 + 双终止条件**
>
> 1. **进入目标格**：回合正常结束（得 $+1$），累积回报有上界；
> 2. **进入禁止格**：回合立即终止（得 $-10$），此后无任何奖励，惩罚信号不被稀释；
> 3. 每回合逆向计算**回合内**的折扣累积回报 $G_t$，$Q$ 值估计真实反映当前策略的质量。

> **附加改进：跨回合累积更新（常数步长 $\alpha$）**
>
> 旧版在每回合开始时重置计数器和累计回报，Q 表被**当前回合的均值完全覆盖**，上一回合的信息就此丢失。  
> 策略持续改善过程中，每一回合的 Q 估计都有较高方差，单次覆盖使 Q 表随每回合样本质量剧烈波动，  
> 无法在多回合间平滑积累——某次偶然的差回合就能把已学好的 Q 值全部冲掉。  
> 改进版改用**常数步长增量更新**：
> $$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl(G_t - Q(s,a)\bigr), \quad \alpha \in (0,1]$$
> Q 表不再每回合重置，所有历史回合的经验均保留在 Q 表中，近期样本权重大、远期样本权重以 $(1-\alpha)^n$ 指数衰减。  
> 策略改善后的正向样本因此能逐步覆盖早期差样本的负面影响，收敛更加稳健。

### 4.1 重新初始化策略与 Q 表

In [5]:
# 重新随机初始化策略，shape=(25, 5)，one-hot 编码，每行恰有一个 1
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
# 初始化 Q 表为全零，shape=(25, 5)，第 0 维为 25 个状态，第 1 维为 5 个动作
qtable = np.zeros((rows * columns, 5))
gridworld.show()              # 打印网格世界布局
gridworld.showPolicy(policy)  # 打印初始随机策略

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
⬆️⬇️➡️➡️⬅️
⬇️⏪⏪⬆️⬅️
⬇️⬅️⏫️⬇️⬆️
⬅️🔄✅⏩️⬅️
⬅️⏩️⬇️➡️🔄


### 4.2 回合制训练循环

> **为何不再用「累计均值」，而改用「增量更新（alpha）」？**
>
> 旧版 Q 更新：$Q(s,a) \leftarrow \dfrac{\sum G}{N}$（所有历史回报的简单均值）
>
> 问题：训练初期 $\varepsilon$ 很大，智能体几乎随机行走，频繁触碰边界（惩罚 $-1$）和禁止区（惩罚 $-10$），
> 积累了大量负回报样本。随着策略改善，后期样本回报为正，但**早期大量坏样本永远拉低均值**，
> 导致某些应为正值的 $Q(s,a)$ 始终估计为负，策略无法收敛到真正最优。
>
> 修正：改用**指数加权增量更新**，每次只对 Q 值做一小步修正：
> $$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl(G_t - Q(s,a)\bigr), \quad \alpha \in (0,1]$$
> $\alpha$（学习率）控制每次更新的步长：$\alpha$ 越大，近期样本权重越大、旧经验衰减越快；
> $\alpha$ 越小，更新越保守。取 $\alpha=0.1$ 在稳定性与更新速度之间取得平衡，在回合数较少时能更快响应策略改善，
> **让策略改善后的正回报样本逐步覆盖早期坏样本的影响**，从而正确收敛。>
> **与增量均值估计公式的关系**
>
> 上述更新公式实质上是**标准增量均值估计公式**的推广——只需将步长 $\dfrac{1}{n}$ 替换为常数 $\alpha$：
>
> $$Q_n = Q_{n-1} + \frac{1}{n}(G_n - Q_{n-1}) \quad\Longrightarrow\quad Q \leftarrow Q + \alpha\bigl(G_t - Q(s,a)\bigr)$$
>
> 展开递推，经过 $n$ 次更新后：
> $$Q_n = (1-\alpha)^n Q_0 + \alpha \sum_{k=1}^{n}(1-\alpha)^{n-k}G_k$$
>
> 第 $k$ 次样本 $G_k$ 的权重为 $\alpha(1-\alpha)^{n-k}$，距当前越远的样本权重越小（指数衰减），越近的样本影响越大：
>
> ---

| | $\frac{1}{n}$ 步长（简单均值） | 常数步长 $\alpha$（指数加权均值） |
|---|---|---|
| 各样本权重 | 等权 $\frac{1}{n}$ | $\alpha(1-\alpha)^{n-k}$，越近越大 |
| 旧样本影响 | 永久保留，不会衰减 | 以 $(1-\alpha)^n$ 指数速率衰减 |
| 适合场景 | 平稳环境（分布不变） | 非平稳环境（策略持续改善） |

> ---
>
> 本节选用常数步长 $\alpha$ 正因训练过程**非平稳**：策略随 $\varepsilon$ 衰减持续改善，近期较好的样本需要快速盖过早期坏样本的影响，而简单均值的旧样本永不衰减，做不到这一点。

In [6]:
np.random.seed(0)
random.seed(0)

# 重置 Q 表和初始策略，不依赖 Cell 11 的随机状态，保证每次运行结果完全一致
qtable = np.zeros((rows * columns, 5))  # Q 表全零初始化，np.ndarray，shape=(状态数 25, 动作数 5)
# Q 表全零时，argmax 对每行返回第一个最大元素的索引 = 0 = 动作“上”
# （平局时的默认选择，并非 MC 算法指定向上），随着训练进行 Q 值将分化并自动修正
policy = np.eye(5)[np.argmax(qtable, axis=1)]  # 初始策略（Q 全零→ argmax=0 → 每个状态默认选动作 0），shape=(25, 5)

# ===== 超参数 =====
# 初始探索率，float；比旧版更大，因为单个回合较短、样本少，需要更强的探索以覆盖全状态空间
epsilon = 0.5
# 总回合数，int；每回合为一条在目标格或禁止格处终止的完整有限轨迹
# ── 回合数推算 ──────────────────────────────────────────────────────
# 旧版总 Q 更新量：400 回合 × 2000 步 = 800,000 次
# 改进版每回合平均步数：~30 步（5×5 网格，遇目标/禁止区即终止）
# 匹配旧版更新量：800,000 ÷ 30 ≈ 26,700 回合
# 但改进版回报更干净（无跨回合污染），配合 alpha=0.2 收敛更快，
# 5,000 回合即可稳定收敛
#   （注：经脚本自动搜索，5 个随机种子全部收敛）
num_episodes = 5000
# 每回合最大步数上限，int；防止在未遇到终止状态时陷入无限循环
max_steps_per_episode = 300  # 每回合最大步数上限，int
# 增量更新学习率 α，float，范围 (0,1]：
#   Q(s,a) ← Q(s,a) + α*(G - Q(s,a))
#   α=0.2 表示每次更新修正当前误差的 20%，比 0.1 收敛快一倍；
#   本环境撞墙返回 -1 且立即终止回合，早期随机策略频繁撞墙，
#   导致通向墙壁的动作 Q 值变负；若 α=0.1，后期正回报样本难以
#   快速覆盖这些负值，远端状态易锁死在停留动作（Q_stay=0 > Q_dir<0）；
#   α=0.2 加快正回报覆盖速度，5,000 回合内可完全收敛
alpha = 0.2

# ε 线性衰减步长，float：前 50% 回合（共 2,500 回合）将 ε 从 0.5 衰减到 0
# 剩余 50%（2,500 回合）维持 ε=0（纯贪婪），配合随机起点精细收敛
# ── 为什么 eps_min=0 仍然收敛 ─────────────────────────────────────────────────
# 距目标最远的状态需约 14 步到达目标，初期 ε 大（0.5→0）时已充分覆盖；
# 配合随机起点（Exploring Starts），后期纯贪婪阶段也能维持全局访问覆盖。
# 若仍担心死锁，可将 eps_min 设为 0.05，但 α=0.2 时通常不必要
epsilon_decay = (epsilon - 0.0) / (num_episodes * 0.5)

for episode in range(num_episodes):  # 按回合循环，episode 为当前回合编号，int，范围 [0, num_episodes)

    # ε 线性衰减，下限为 0.0，后期退化为纯贪婪策略，配合随机起点保证全局覆盖
    epsilon = max(0.0, epsilon - epsilon_decay)  # float，逐回合减小

    # 最优动作选择概率，float：p1 = 1 - ε*(|A|-1)/|A|，|A|=5
    p1 = 1 - epsilon * (4 / 5)
    # 非最优动作选择概率，float：p0 = ε/|A|，均匀分配给各非最优动作
    p0 = epsilon / 5
    # 构建 one-hot 值（0/1）到 ε-greedy 概率的映射字典，dict{int: float}
    d = {1: p1, 0: p0}
    # 将 one-hot policy 转为 ε-greedy 概率策略，shape=(25, 5)，每行为合法概率分布（行和=1）
    policy_epsilon = np.vectorize(d.get)(policy)

    # === 采集一个完整回合的轨迹 ===
    # 随机起始状态（Exploring Starts），int，范围 [0, 24]
    nowState  = random.randint(0, rows * columns - 1)
    # 随机起始动作（Exploring Starts），int，范围 [0, 4]
    nowAction = random.randint(0, 4)

    # 存储本回合轨迹，list[tuple(int, int, float)]：(状态编号, 动作编号, 即时奖励)
    ep_data = []

    for _ in range(max_steps_per_episode):  # 每步上限 max_steps_per_episode
        # 执行动作，获取即时奖励（float）和下一状态编号（int）
        reward, nextState = gridworld.getScore(nowState, nowAction)
        ep_data.append((nowState, nowAction, reward))  # 记录当前步信息

        # === 双终止条件：进入目标格（reward==score）或禁止格（reward==forbiddenAreaScore）===
        # reward != 0 等价于进入了非普通格（目标格 +1 或禁止格 -10），回合立即结束
        # 关键：禁止格终止后无后续奖励，-10 的惩罚不会被后续 +1 稀释，信号清晰
        if reward != 0:
            break  # 结束本回合，不再继续采样

        # 按 ε-greedy 策略在下一状态选择动作
        # np.random.choice(5, p=...)：从 [0,1,2,3,4] 按概率分布 policy_epsilon[nextState] 采样 1 个动作
        # 返回 int，范围 [0, 4]
        nowAction = np.random.choice(5, p=policy_epsilon[nextState])
        nowState  = nextState  # 更新当前状态

    # === Every-Visit MC：逆向递推回合内折扣累积回报 G，用 alpha 增量更新 Q 表 ===
    # 从回合末端向前逐步递推：G_t = r_t + γ·G_{t+1}
    # 关键区别：G 只在本回合范围内累积，不跨回合叠加，回报估计无偏且干净
    G = 0.0  # 折扣累积回报初始值，float，从回合末端（G_{T}=0）开始向前递推
    for state, action, r in reversed(ep_data):  # 逆序遍历本回合轨迹
        G = r + gamma * G  # G_t = r_t + γ·G_{t+1}，float
        # 增量更新：Q(s,a) ← Q(s,a) + α*(G - Q(s,a))，float
        # 近期样本（策略改善后）的权重大于早期坏样本，避免均值被污染
        # Every-Visit MC：同一 (s,a) 对在本回合出环 k 次就更新 k 次
        #   每次的 G 均从该时刻向回合末端递推，实际影响微小
        #   （本回合平均 ~30 步，125 个 (s,a) 对单回合重复访问率很低）
        qtable[state][action] += alpha * (G - qtable[state][action])

    # === 策略改进：每回合后立即贪婪更新策略 ===
    # np.argmax(qtable, axis=1)：对每个状态取 Q 值最大的动作索引，shape=(25,)，dtype=int
    # np.eye(5)[...]：花式索引，将整数动作索引转为 one-hot 编码，shape=(25, 5)
    policy = np.eye(5)[np.argmax(qtable, axis=1)]

    pass  # 训练过程中不打印，不调用 clear_output，减少 I/O 开销，训练速度提升 10~20 倍

# === 训练完成后统一打印最终结果 ===
# 基于最终 ε 重建 ε-greedy 概率策略，用于计算状态价值，shape=(25, 5)
p1_final = 1 - epsilon * (4 / 5)  # 最优动作的选择概率，float
p0_final = epsilon / 5             # 非最优动作的选择概率，float
policy_epsilon = np.vectorize({1: p1_final, 0: p0_final}.get)(policy)  # shape=(25, 5)
# V(s) = Σ_a π(a|s)·Q(s,a)，list[float]，长度 25
values = [
    sum(policy_epsilon[s][a] * qtable[s][a] for a in range(5))
    for s in range(rows * columns)
]
print(f"\n{'*' * 55}")
print(f"  改进版训练完成，共训练 {num_episodes} 回合")
print(f"  最终 ε={epsilon:.4f}  |  状态价值均值：{np.mean(values):.6f}")
# 打印状态价值矩阵，np.ndarray，shape=(5, 5)
print(np.array(values).reshape(rows, columns))
gridworld.showPolicy(policy)  # 可视化最终贪婪策略
print(f"{'*' * 55}")


*******************************************************
  改进版训练完成，共训练 5000 回合
  最终 ε=0.0000  |  状态价值均值：0.600207
[[0.34867844 0.38742049 0.43046721 0.4782969  0.531441  ]
 [0.3138106  0.32296057 0.47186672 0.531441   0.59049   ]
 [0.28242954 0.25418658 0.99999269 0.59049    0.6561    ]
 [0.25418658 0.99986708 0.99974039 0.99989366 0.729     ]
 [0.22876748 0.89365073 1.         0.9        0.81      ]]
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️⬇️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
*******************************************************


### 4.3 最终策略可视化

In [7]:
gridworld.show()              # 以 emoji 打印网格世界布局，无返回值
gridworld.showPolicy(policy)  # 可视化修正版训练后的最终策略，无返回值

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
➡️➡️➡️➡️⬇️
⬆️⏫️⏩️⬇️⬇️
⬆️⬅️⏬➡️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
